This is a markdown cell. It describes the purpose of this notebook and its code.

This notebook will translate the Wiltshire records from Latin to English using ChatGPT.

Last updated by Kuba Kowalski on 04/03/2026, 15:30.

In [1]:
# Import packages
import os
import subprocess
from pathlib import Path
import pdfplumber
from docx import Document
import fitz  # PyMuPDF

# ChatGPT appraoch

In [1]:
!pip install -U openai python-docx pymupdf

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------- -------------------------------- 0.2/1.1 MB 6.3 MB/s eta 0:00:01
   ----------------------------- ---------- 0.8/1.1 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 11.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
    --------------------------------------- 0.4/19.2 MB 22.4 MB/s eta 0:00:01
    --------------------------------------- 0.4/19.2 MB 22.4 MB/s eta 0:00:01
   - -------------------------------------- 0.5/19.2 MB 3.7 MB/s eta 0:00:06
   -- ------------------------------------- 1.2/19.2 MB 6.2 MB/s eta 0:00:03
   ---- ----------------------------------- 1.9/19.2 MB 8.2 MB/s eta 0:00:03
   ----- ---------------------------------- 2.7/19.2 MB 9.6 MB/s eta 0:00:02
   ------- -------------------------------- 3.5/19.2 MB 10.6 MB/s eta 0:00:02
   -------- ------------------------------- 3.9/19.2 MB 10.5 MB/s eta 0:00:02
   --------

  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# OCR PDFs to Word

In [5]:
# OCR all PDFs in a folder 

import base64
from pathlib import Path
from datetime import datetime
import fitz
from docx import Document
from docx.shared import Pt
import re
from openai import OpenAI

# ----------------------------
# CONFIG
# ----------------------------
API_KEY_PATH = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\open-ai-key.txt")
INPUT_DIR = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\Records\chapters")
OUTPUT_DIR = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\OpenAI OCR Wiltshire")

MODEL = "gpt-4.1" # Cheapest option, still good enough. Any performance improvements are best done using the prompt
RENDER_SCALE = 2.5
MAX_OUTPUT_TOKENS = 6000
OVERWRITE = False   # set True to re-run existing files

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = API_KEY_PATH.read_text(encoding="utf-8").strip()
if not API_KEY:
    raise SystemExit("API key missing")

client = OpenAI(api_key=API_KEY)

tag_re = re.compile(r"^<(H|P)>(.*)</\1>\s*$")

# ----------------------------
# HELPERS
# ----------------------------
def pdf_page_to_base64_png(pdf_path, page_num):
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(matrix=fitz.Matrix(RENDER_SCALE, RENDER_SCALE), alpha=False)
    img = pix.tobytes("png")
    doc.close()
    return base64.b64encode(img).decode("utf-8")

def ocr_page(image_b64, page_num, total_pages):
    prompt = (
        "Transcribe the MAIN BODY text from this scanned MEDIEVAL LATIN page.\n\n"
        "EXCLUDE: page numbers, headers, footnotes.\n\n"
        "OUTPUT FORMAT:\n"
        "- Section headings → <H>...</H>\n"
        "- Other lines → <P>...</P>\n"
        "- Preserve original spelling, punctuation, capitalization, numbers, and line breaks exactly as written.\n"
        "- Do NOT translate or modernize Latin.\n"
        "- Output ONLY tagged lines.\n"
    )

    resp = client.responses.create(
        model=MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"data:image/png;base64,{image_b64}"}
            ]
        }],
        max_output_tokens=MAX_OUTPUT_TOKENS
    )

    print(f"    OCR Page {page_num+1}/{total_pages} ✓")
    return (resp.output_text or "").strip()

def write_docx(pdf_name, pages, out_file):
    doc = Document()
    doc.add_paragraph(f"{pdf_name} — OCR output ({datetime.now().strftime('%Y-%m-%d %H:%M')})")

    for i, page_text in enumerate(pages, 1):
        doc.add_paragraph(f"--- Page {i} ---")

        for line in page_text.splitlines():
            line = line.strip()
            if not line:
                continue

            m = tag_re.match(line)
            if m:
                tag, content = m.group(1), m.group(2).strip()
                p = doc.add_paragraph()
                run = p.add_run(content)
                if tag == "H":
                    run.bold = True
                    run.font.size = Pt(12)
            else:
                doc.add_paragraph(line)

        if i < len(pages):
            doc.add_page_break()

    doc.save(out_file)

def process_pdf(pdf_path):
    out_file = OUTPUT_DIR / f"{pdf_path.stem}_ocr.docx"

    if out_file.exists() and not OVERWRITE:
        print(f"SKIP (exists): {pdf_path.name}")
        return

    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()

    print(f"\nProcessing {pdf_path.name} ({total_pages} pages)")

    pages = []
    for i in range(total_pages):
        img = pdf_page_to_base64_png(pdf_path, i)
        pages.append(ocr_page(img, i, total_pages))

    write_docx(pdf_path.name, pages, out_file)
    print(f"Saved: {out_file}")

# ----------------------------
# RUN ALL
# ----------------------------
pdfs = sorted(INPUT_DIR.glob("*.pdf"))
if not pdfs:
    raise SystemExit(f"No PDFs found in {INPUT_DIR}")

print(f"Found {len(pdfs)} PDFs")

for pdf in pdfs:
    process_pdf(pdf)

print("\nDone.")


Found 24 PDFs

Processing extents1271-1277.pdf (44 pages)
    OCR Page 1/44 ✓
    OCR Page 2/44 ✓
    OCR Page 3/44 ✓
    OCR Page 4/44 ✓
    OCR Page 5/44 ✓
    OCR Page 6/44 ✓
    OCR Page 7/44 ✓
    OCR Page 8/44 ✓
    OCR Page 9/44 ✓
    OCR Page 10/44 ✓
    OCR Page 11/44 ✓
    OCR Page 12/44 ✓
    OCR Page 13/44 ✓
    OCR Page 14/44 ✓
    OCR Page 15/44 ✓
    OCR Page 16/44 ✓
    OCR Page 17/44 ✓
    OCR Page 18/44 ✓
    OCR Page 19/44 ✓
    OCR Page 20/44 ✓
    OCR Page 21/44 ✓
    OCR Page 22/44 ✓
    OCR Page 23/44 ✓
    OCR Page 24/44 ✓
    OCR Page 25/44 ✓
    OCR Page 26/44 ✓
    OCR Page 27/44 ✓
    OCR Page 28/44 ✓
    OCR Page 29/44 ✓
    OCR Page 30/44 ✓
    OCR Page 31/44 ✓
    OCR Page 32/44 ✓
    OCR Page 33/44 ✓
    OCR Page 34/44 ✓
    OCR Page 35/44 ✓
    OCR Page 36/44 ✓
    OCR Page 37/44 ✓
    OCR Page 38/44 ✓
    OCR Page 39/44 ✓
    OCR Page 40/44 ✓
    OCR Page 41/44 ✓
    OCR Page 42/44 ✓
    OCR Page 43/44 ✓
    OCR Page 44/44 ✓
Saved: C:\Users\kubak\OneDr

In [6]:
import re
from pathlib import Path
from docx import Document

# ----------------------------
# INPUT/OUTPUT
# ----------------------------
input_dir = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\OpenAI OCR Wiltshire")

translation_dir = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\OpenAI Translation Wiltshire")
translation_dir.mkdir(parents=True, exist_ok=True)

page_sep_re = re.compile(r"^\s*---\s*Page\s*(\d+)\s*---\s*$", re.IGNORECASE)

SYSTEM_PROMPT = """
You are a professional translator of medieval Latin manorial accounts into modern English.

Your job is to translate the ENTIRE provided text from beginning to end. Do not compress any text.

You MUST translate the full text completely.
You MUST NOT summarize, shorten, omit, condense, or paraphrase.
You MUST NOT stop early.
You MUST continue until the final line of the input is translated.

Your output must correspond line-by-line to the input.
Every line of Latin must produce a translated English line.

Produce ENGLISH TEXT ONLY, except for proper names and ORIGINAL NUMBER EXPRESSIONS KEPT IN BRACKETS.

GENERAL RULES

1. Translate ALL Latin into clear, modern English.
   - Do NOT leave Latin sentences in the output.
   - The only Latin that may remain are:
     - proper names (people, places, e.g. Ricardus, Cuxham, Oxon'),
     - the original Latin number expressions, which MUST be preserved in square brackets after the translation of the number.

2. Section labels like "Redditus", "Exitus Manerii", "Instaurum", "Dragetum", "Fabe", "Pise", "Auena", etc. must be translated into English
   (you may optionally keep the Latin in brackets, e.g. "Receipts (Redditus)").

NUMBER AND MONEY RULES (VERY IMPORTANT)

3. Whenever you encounter any Latin numerical expression (Roman numerals or medieval Arabic forms with abbreviations),
   you MUST convert it to Arabic numerals in English AND immediately follow it with the original Latin in square brackets.

   FORMAT:
   <Arabic value(s) in English> <unit(s)> [<original Latin number phrase>]

   Examples:
   - "xijs. v d. ob." -> "12 shillings 5.5 pence [xij s. v d. ob.]"
   - "xxvij s."       -> "27 shillings [xxvij s.]"
   - "xxij porcis"    -> "22 pigs [xxij porcis]"
   - "vij qr. Frumenti" -> "7 quarters of wheat [vij qr. Frumenti]"

4. Monetary units:
   - Translate "libra" (l.) as "pounds",
     "solidi" (s.) as "shillings",
     "denarii" (d.) as "pence",
     "obolus" (ob.) as 0.5 pence.
   - Do NOT change the numeric values themselves, only convert to Arabic numerals.
   - Always append the full original Latin monetary expression in brackets after your English rendering.

5. For non-monetary quantities (quarters, bushels, acres, animals, etc.), follow the same pattern.

6. If a number appears already as Arabic digits in the Latin text, you may keep it as is.
   If the Latin includes a Roman or medieval written number phrase, you MUST preserve that phrase in brackets.

7. Convert all monetary fractions into decimal form.
   obolus (ob.) = 0.5 pence
   q (quadrans) = 0.25 if it appears
   q. or qr. = "quarters" (grain measure), not fractions.

STRUCTURE AND FORMATTING

8. Preserve page markers like [Page 3] exactly as they are.

9. If a line is fully in English, keep it unchanged.
   If a line mixes English and Latin, translate only the Latin parts.

10. OUTPUT FORMAT:
   - Plain text only.
   - No markdown, no bold, no lists.
   - Maintain the original line structure.
   - Do NOT omit any lines.
   - Translate until the end of the provided text.
"""

# ----------------------------
# Functions
# ----------------------------
def read_docx_as_text(docx_path: Path) -> str:
    doc = Document(docx_path)
    lines = []
    for p in doc.paragraphs:
        t = (p.text or "").rstrip()
        if not t:
            lines.append("")
            continue

        m = page_sep_re.match(t)
        if m:
            lines.append(f"[Page {m.group(1)}]")
        else:
            lines.append(t)

    return "\n".join(lines).strip()

def translate_document(text_block):
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text_block},
        ],
    )
    return response.output_text

def write_translation_docx(text: str, out_path: Path) -> None:
    doc = Document()
    for line in text.splitlines():
        doc.add_paragraph(line)
    doc.save(out_path)

# ----------------------------
# PROCESS ALL DOCX IN INPUT DIRECTORY
# ----------------------------
docx_files = sorted(input_dir.glob("*.docx"))

if not docx_files:
    raise SystemExit(f"No DOCX files found in {input_dir}")

print(f"Found {len(docx_files)} DOCX file(s).")

for docx_file in docx_files:
    out_docx = translation_dir / f"{docx_file.stem}_EN.docx"
    out_txt  = translation_dir / f"{docx_file.stem}_EN.txt"

    if out_docx.exists():
        print(f"SKIP (already translated): {docx_file.name}")
        continue

    print(f"\nTranslating: {docx_file.name}")

    text_block = read_docx_as_text(docx_file)
    print("Characters:", len(text_block))

    translated = translate_document(text_block)

    write_translation_docx(translated, out_docx)
    out_txt.write_text(translated, encoding="utf-8")

    print(f"Saved: {out_docx}")

print("\nDone.")

Found 24 DOCX file(s).

Translating: extents1271-1277_ocr.docx
Characters: 92651
Saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\OpenAI Translation Wiltshire\extents1271-1277_ocr_EN.docx

Translating: particulars_of_account_for_building_a_barn_in_sevenhampton1280_ocr.docx
Characters: 1876
Saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\OpenAI Translation Wiltshire\particulars_of_account_for_building_a_barn_in_sevenhampton1280_ocr_EN.docx

Translating: sevenhampton1269-1270_ocr.docx
Characters: 18421
Saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\OpenAI Translation Wiltshire\sevenhampton1269-1270_ocr_EN.docx

Translating: sevenhampton1272-1273_ocr.docx
Characters: 23678
Saved: C:\Users\kubak\OneDrive - Wageningen University & Res

# Translation from Latin to English using ChatGPT

In [ ]:
import re
from pathlib import Path
from docx import Document

# ----------------------------
# INPUT/OUTPUT
# ----------------------------
input_dir = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\Records\chapters")

translation_dir = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Wiltshire\OpenAI Translation Wiltshire")
translation_dir.mkdir(parents=True, exist_ok=True)

page_sep_re = re.compile(r"^\s*---\s*Page\s*(\d+)\s*---\s*$", re.IGNORECASE)

SYSTEM_PROMPT = """
You are a professional translator of medieval Latin manorial accounts into modern English.

Your job is to translate the ENTIRE provided text from beginning to end. Do not compress any text.

You MUST translate the full text completely.
You MUST NOT summarize, shorten, omit, condense, or paraphrase.
You MUST NOT stop early.
You MUST continue until the final line of the input is translated.

Your output must correspond line-by-line to the input.
Every line of Latin must produce a translated English line.

Produce ENGLISH TEXT ONLY, except for proper names and ORIGINAL NUMBER EXPRESSIONS KEPT IN BRACKETS.

GENERAL RULES

1. Translate ALL Latin into clear, modern English.
   - Do NOT leave Latin sentences in the output.
   - The only Latin that may remain are:
     - proper names (people, places, e.g. Ricardus, Cuxham, Oxon'),
     - the original Latin number expressions, which MUST be preserved in square brackets after the translation of the number.

2. Section labels like "Redditus", "Exitus Manerii", "Instaurum", "Dragetum", "Fabe", "Pise", "Auena", etc. must be translated into English
   (you may optionally keep the Latin in brackets, e.g. "Receipts (Redditus)").

NUMBER AND MONEY RULES (VERY IMPORTANT)

3. Whenever you encounter any Latin numerical expression (Roman numerals or medieval Arabic forms with abbreviations),
   you MUST convert it to Arabic numerals in English AND immediately follow it with the original Latin in square brackets.

   FORMAT:
   <Arabic value(s) in English> <unit(s)> [<original Latin number phrase>]

   Examples:
   - "xijs. v d. ob." -> "12 shillings 5.5 pence [xij s. v d. ob.]"
   - "xxvij s."       -> "27 shillings [xxvij s.]"
   - "xxij porcis"    -> "22 pigs [xxij porcis]"
   - "vij qr. Frumenti" -> "7 quarters of wheat [vij qr. Frumenti]"

4. Monetary units:
   - Translate "libra" (l.) as "pounds",
     "solidi" (s.) as "shillings",
     "denarii" (d.) as "pence",
     "obolus" (ob.) as 0.5 pence.
   - Do NOT change the numeric values themselves, only convert to Arabic numerals.
   - Always append the full original Latin monetary expression in brackets after your English rendering.

5. For non-monetary quantities (quarters, bushels, acres, animals, etc.), follow the same pattern.

6. If a number appears already as Arabic digits in the Latin text, you may keep it as is.
   If the Latin includes a Roman or medieval written number phrase, you MUST preserve that phrase in brackets.

7. Convert all monetary fractions into decimal form.
   obolus (ob.) = 0.5 pence
   q (quadrans) = 0.25 if it appears
   q. or qr. = "quarters" (grain measure), not fractions.

STRUCTURE AND FORMATTING

8. Preserve page markers like [Page 3] exactly as they are.

9. If a line is fully in English, keep it unchanged.
   If a line mixes English and Latin, translate only the Latin parts.

10. OUTPUT FORMAT:
   - Plain text only.
   - No markdown, no bold, no lists.
   - Maintain the original line structure.
   - Do NOT omit any lines.
   - Translate until the end of the provided text.
"""

# ----------------------------
# Functions
# ----------------------------
def read_docx_as_text(docx_path: Path) -> str:
    doc = Document(docx_path)
    lines = []
    for p in doc.paragraphs:
        t = (p.text or "").rstrip()
        if not t:
            lines.append("")
            continue

        m = page_sep_re.match(t)
        if m:
            lines.append(f"[Page {m.group(1)}]")
        else:
            lines.append(t)

    return "\n".join(lines).strip()

def translate_document(text_block):
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text_block},
        ],
    )
    return response.output_text

def write_translation_docx(text: str, out_path: Path) -> None:
    doc = Document()
    for line in text.splitlines():
        doc.add_paragraph(line)
    doc.save(out_path)

# ----------------------------
# PROCESS ALL DOCX IN INPUT DIRECTORY
# ----------------------------
docx_files = sorted(input_dir.glob("*.docx"))

if not docx_files:
    raise SystemExit(f"No DOCX files found in {input_dir}")

print(f"Found {len(docx_files)} DOCX file(s).")

for docx_file in docx_files:
    out_docx = translation_dir / f"{docx_file.stem}_EN.docx"
    out_txt  = translation_dir / f"{docx_file.stem}_EN.txt"

    if out_docx.exists():
        print(f"SKIP (already translated): {docx_file.name}")
        continue

    print(f"\nTranslating: {docx_file.name}")

    text_block = read_docx_as_text(docx_file)
    print("Characters:", len(text_block))

    translated = translate_document(text_block)

    write_translation_docx(translated, out_docx)
    out_txt.write_text(translated, encoding="utf-8")

    print(f"Saved: {out_docx}")

print("\nDone.")

Found 2 DOCX file(s).

Translating: Cuxham_1276_ocr.docx
Characters: 10553
Saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Cuxham\OpenAI Translation Cuxham\Cuxham_1276_ocr_EN.docx

Translating: Cuxham_1288_ocr.docx
Characters: 21730
Saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Latin\Cuxham\OpenAI Translation Cuxham\Cuxham_1288_ocr_EN.docx

Done.
